# A project's TRE reads the catalog of a Data Provider

**Run it interactively, in a notebook of your own JupyterHub pod in the TRE of your project (`cvd-2026` or `diab-2026`), signed in to the hub as a member of the project, or by `smoke-notebook`, which signs the project's smoke member in to the hub and runs it in their pod** (it is in `project-notebooks.csv`, which `smoke-notebook` reads in a project). `OIDC_TOKEN` is in the pod
after the sign-in, whoever signs in. To run it yourself, copy it in with `afon-cli.sh manage-project-stack run <project> kube-user-cp --user <you> to <path>`, or upload it in JupyterLab, and run all the cells.

It checks, one step at a time, what a member needs to read the catalog of the project's Data Provider (`cvd` of `lon` for `cvd-2026`, `diab` of `nw` for `diab-2026`),
and says which step fails, so that the failure says where to look:

1. the token that the notebook holds names the project, the Data Provider as an audience, and the second factor;
2. the notebook pod cannot call the Data Provider's Polaris directly, as the TRE's whitelist has it: only Spark can;
3. Spark reads the table that the Data Provider has released, and gets its columns;
4. Spark is refused a table that the Data Provider has not released.

**The project is that of the member's token, and the Data Provider is that of the project**: nothing is edited to run it as a member of either TRE. Add a project and its Data Provider
to `PROJECTS` in the first cell when one is made.

**To test that a member cannot read another project's Data Provider**, set `TARGET` (or `AFON_TARGET`) to the other project, eg `diab-2026` in a pod of `cvd-2026`, and run it again. The member's token
is then used against the other project's Data Provider, and the steps pass only if it is refused.

Needs two tables on each Data Provider, which its own smoke test writes (`afon-cli.sh manage-project-stack run <data project> spark-polaris-smoke`): `<catalog>.staging.smoke_test`, which
the Data Provider then releases (`afon-cli.sh manage-project-stack run <data project> manage-data release <catalog> smoke_test`), and `<catalog>.staging.smoke_test_unreleased`, which stays in `staging`.

In [ ]:
# The Data Provider of each project, as its catalog is in the TRE's config: the audience of its tokens, the endpoint of its Polaris (AZURE_CATALOG_URI_<id>) and the name of the catalog.
PROJECTS = {
    "cvd-2026":  {"data_provider": "lon", "audience": "provider-lon", "uri": "https://azure-polaris.lon.emrys2.click/api/catalog", "catalog": "cvd"},
    "diab-2026": {"data_provider": "nw",  "audience": "provider-nw",  "uri": "https://azure-polaris.nw.emrys2.click/api/catalog",  "catalog": "diab"},
}

import os, json, base64, time

token = os.environ.get("OIDC_TOKEN")
claims = {}
if token:
    payload = token.split(".")[1]
    claims = json.loads(base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)))
claim_project = claims.get("project") or []
claim_projects = [claim_project] if isinstance(claim_project, str) else list(claim_project)

# The project of the member is that of their token: set AFON_PROJECT only if it names none.
PROJECT = os.environ.get("AFON_PROJECT") or (claim_projects[0] if claim_projects else None)
# The project whose Data Provider is read: the member's own, unless AFON_TARGET names another project.  Another project's Data Provider is what the member
# must be REFUSED, so the steps then pass when the read is refused: that is the test that a project cannot read what is not its own.  To try it, set
# it here, eg TARGET = "diab-2026" in a pod of cvd-2026, or set AFON_TARGET in the environment of the notebook.
TARGET = os.environ.get("AFON_TARGET") or PROJECT
CROSS = TARGET != PROJECT

for name, value in (("PROJECT", PROJECT), ("TARGET", TARGET)):
    if value not in PROJECTS:
        raise SystemExit(f"{name} is {value!r}, which is not a project known here ({sorted(PROJECTS)}): is OIDC_TOKEN set, from the sign-in, and does it name the project? "
                         "Add the project to PROJECTS, or set AFON_PROJECT.")

DATA_PROVIDER = PROJECTS[TARGET]["data_provider"]
AUDIENCE      = PROJECTS[TARGET]["audience"]                     # the audience of the Data Provider
CATALOG_URI   = PROJECTS[TARGET]["uri"]
CATALOG_NAME  = PROJECTS[TARGET]["catalog"]                      # the name of the catalog in the Data Provider's Polaris
SPARK_CATALOG = f"test_{DATA_PROVIDER}_{CATALOG_NAME}"           # what it is called here
RELEASED      = "released"                                       # the namespace that the project may read
NOT_RELEASED  = "staging"                                        # the namespace that it may not
TABLE         = "smoke_test"                                     # released by the Data Provider
UNRELEASED_TABLE = "smoke_test_unreleased"                       # made in NOT_RELEASED, and not released: it must exist there
EXPECTED_COLUMNS = ["value"]
SPARK_CONNECT = os.environ.get("SPARK_REMOTE", "sc://spark-connector-0-driver-svc:15002").split(";")[0].rstrip("/")   # the address only: in a pod, SPARK_REMOTE has the token after it, which is put here; from a machine, it is the port-forward, which notebook.sh makes

results = {}

def record(step, ok, detail=""):
    """ok is True (pass), False (fail), or None: not conclusive, which the result says but does not count as a failure"""
    results[step] = ok
    label = "PASS" if ok is True else "FAIL" if ok is False else "INCONCLUSIVE"
    print(label, "-", step, ("- " + detail) if detail else "")


# What a refusal of authorization says, from the Data Provider's Polaris
DENIED = ("forbidden", "not authorized", "notauthorized", "403", "permission", "not allowed", "denied")

def layer_of(text):
    """Which layer refused, as far as the answer says: an HTML page is the Data Provider's ingress, JSON or Iceberg's own wording is its Polaris"""
    return "by the Data Provider's ingress (an HTML page: the TRE's address is not admitted)" if "<html" in text.lower() else "by the Data Provider's Polaris"

print(f"The member is in {PROJECT}, and reads the Data Provider {DATA_PROVIDER} of {TARGET}, catalog {CATALOG_NAME}"
      + (": ANOTHER project's, which must refuse it" if CROSS else ""))

## 1. The token

`OIDC_TOKEN` is put in the environment of the notebook when it starts, from the member's sign-in.  Only the claims are printed, never the token.

In [ ]:
if not token:
    record("token is in the environment", False, "OIDC_TOKEN is not set: sign out and in again, or the hub does not pass it on")
else:
    for name in ("iss", "aud", "azp", "acr", "project", "polaris_principal", "email"):
        print(f"{name} = {claims.get(name)}")
    remaining = claims["exp"] - time.time()
    record("token names the project", PROJECT in claim_projects, f"project = {claims.get('project')}")
    if CROSS:
        print(f"aud has {AUDIENCE}: {AUDIENCE in (claims.get('aud') or [])}, not asserted, as {TARGET}'s Data Provider is not this project's to read")
    else:
        record("token is for the Data Provider", AUDIENCE in (claims.get("aud") or []), f"aud = {claims.get('aud')}")
    record("token says a second factor was used", claims.get("acr") == "mfa", f"acr = {claims.get('acr')}")
    record("token has not expired", remaining > 60, f"{int(remaining)} seconds left")

## 2. The notebook cannot call the Data Provider directly

A member reaches the Data Provider's catalog through Spark, whose driver and workers are allowed its hosts by the TRE's whitelist; the notebook pod is not: its
network policy admits only Spark.  So a call of the catalog API from here, outside Spark, must not get through.  Not resolving the name, or not connecting, is
the result that is wanted: if the call is answered, the notebook has a way out that the whitelist does not mean it to have.

In [ ]:
import urllib.request, urllib.error, socket
from urllib.parse import urlparse

STEP = "the notebook pod cannot call the Data Provider directly"
host = urlparse(CATALOG_URI).hostname
address = None
if not os.environ.get("KUBERNETES_SERVICE_HOST"):
    # run from a machine by smoke-notebook: the network policy is of the notebook pods, so it says nothing of a call from here
    record(STEP, None, "run from outside the cluster, not in a notebook pod, so the network policy of the pods does not apply: network-smoke tests it from a member's pod")
else:
    try:
        address = socket.gethostbyname(host)
    except OSError as e:
        record(STEP, True, f"{host} does not resolve from here, as the network policy of the notebook pods has it: {e}")

if address:
    request = urllib.request.Request(f"{CATALOG_URI}/v1/config?warehouse={CATALOG_NAME}", headers={"Authorization": "Bearer " + token})
    try:
        with urllib.request.urlopen(request, timeout=20) as response:
            record(STEP, False, f"HTTP {response.status}: the notebook reached the Data Provider's Polaris, which its network policy should not allow")
    except urllib.error.HTTPError as e:
        record(STEP, False, f"HTTP {e.code}: the notebook reached the Data Provider's Polaris, which its network policy should not allow")
    except (urllib.error.URLError, socket.timeout, OSError) as e:
        record(STEP, True, f"{host} resolves, and the connection does not get through, as the network policy of the notebook pods has it: {e}")

## 3. Spark session with the Data Provider's catalog

The shared Spark Connect server registers no catalog of its own: each catalog is given here, with the member's token, as the generated session code does.

In [ ]:
from pyspark.sql import SparkSession

builder = (SparkSession.builder.appName("tre-reads-data-provider").remote(SPARK_CONNECT + "/;x-afon-token=" + token)
    .config(f"spark.sql.catalog.{SPARK_CATALOG}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{SPARK_CATALOG}.type", "rest")
    .config(f"spark.sql.catalog.{SPARK_CATALOG}.uri", CATALOG_URI)
    .config(f"spark.sql.catalog.{SPARK_CATALOG}.warehouse", CATALOG_NAME)
    .config(f"spark.sql.catalog.{SPARK_CATALOG}.token", token)
    .config(f"spark.sql.catalog.{SPARK_CATALOG}.cache-enabled", "false")
    .config(f"spark.sql.catalog.{SPARK_CATALOG}.header.X-Iceberg-Access-Delegation", "vended-credentials"))
try:
    spark = builder.getOrCreate()
    record("Spark session is made", True)
except Exception as e:
    record("Spark session is made", False, str(e)[:300])

## 4. Reading what the Data Provider has released

In [ ]:
table = f"{SPARK_CATALOG}.{RELEASED}.{TABLE}"
if CROSS:
    step = f"is refused {table}, of {TARGET}'s Data Provider"
    try:
        spark.sql(f"SELECT * FROM {table}").collect()
        record(step, False, f"it was read: {PROJECT} can read the Data Provider of {TARGET}")
    except Exception as e:
        message = str(e)
        first = message.strip().splitlines()[0][:200] if message.strip() else type(e).__name__
        if any(word in message.lower() for word in DENIED):
            record(step, True, "refused " + layer_of(message) + ": " + first)
        else:
            record(step, None, "an error that is not one of authorization (the host not reached, or the session not made?): " + first)
else:
    try:
        frame = spark.sql(f"SELECT * FROM {table}")
        frame.printSchema()
        rows = frame.collect()
        record(f"reads {table}", True, f"{len(rows)} row(s)")
        record("the table has the columns that were made", frame.columns == EXPECTED_COLUMNS, f"columns = {frame.columns}")
    except Exception as e:
        record(f"reads {table}", False, str(e)[:600])

## 5. Being refused what the Data Provider has not released

The project's role is read-only on `released` and has nothing on `staging`, where the Data Provider makes and checks its tables.  `UNRELEASED_TABLE` is a
table that exists there and has not been released, so reading it must be refused, and the refusal must be one of authorization: a table that is not found
says nothing, as it may not be there, or Polaris may hide it from a caller who has no right to it, and is reported as inconclusive.

In [ ]:
table = f"{SPARK_CATALOG}.{NOT_RELEASED}.{UNRELEASED_TABLE}"
try:
    spark.sql(f"SELECT * FROM {table}").collect()
    record(f"is refused {table}", False, "it was read: the project can read what the Data Provider has not released")
except Exception as e:
    message = str(e)
    first = message.strip().splitlines()[0][:200]
    if any(word in message.lower() for word in DENIED):
        record(f"is refused {table}", True, "refused " + layer_of(message) + ": " + first)
    elif "not found" in message.lower() or "table_or_view_not_found" in message.lower():
        record(f"is refused {table}", None, "not found, which is not a refusal: check that the table exists in " + NOT_RELEASED + " on the Data Provider, or that Polaris hides it: " + first)
    else:
        record(f"is refused {table}", None, "an error that is not one of authorization: " + first)

## Result

In [ ]:
print()
for step, ok in results.items():
    print("PASS" if ok is True else "FAIL" if ok is False else "INCONCLUSIVE", "-", step)
print()
if CROSS:
    print(f"CROSS-PROJECT TEST: a member of {PROJECT} reading the Data Provider of {TARGET}. PASS here means the member was REFUSED, as it must be.")
if any(ok is False for ok in results.values()):
    print("SOMETHING FAILED: the first FAIL above says where" + (" (a FAIL here means the member was NOT refused)" if CROSS else ""))
elif any(ok is None for ok in results.values()):
    print("NO FAILURE, BUT NOT EVERYTHING WAS CONCLUDED: see INCONCLUSIVE")
else:
    print("ALL PASS" + (": the member was refused" if CROSS else ""))

# the notebook fails when a step does, so that a run from a machine, by smoke-notebook, fails too: INCONCLUSIVE does not
failed = [step for step, ok in results.items() if ok is False]
if failed:
    raise AssertionError(f"{len(failed)} of {len(results)} steps failed, the first being: {failed[0]}")